# SAM Auto-Segmentation — pure-SAM method (stage 1 of 2)   ·  Colab / GPU

**Method: SAM (no geometric prior).** First stage of the **pure-SAM** branch of the
three-method comparison. Where the *geometric* method seeds rooms with a deterministic
watershed and *geometric+SAM* refines that watershed with prompted SAM, this method runs
**SAM in automatic "segment everything" mode** directly on the Stage-1 rasters — **no
watershed prior, no prompts** — and turns the resulting masks into room labels
(`scan2bim.segment_rooms_sam_auto`). This reproduces the paper's room pipeline (Albadri et
al., ISPRS Archives XLVIII-G-2025-131, §3.1).

> **Three-method comparison.** geometric (watershed only) · **SAM (this branch)** ·
> geometric+SAM (watershed prior + prompted-SAM refinement). All three write a
> `room_labels.npy` on the **same Stage-1 grid**, so `evaluation/pq_eval.ipynb` scores them
> against the same ground truth with zero resampling.

**GPU stage.** SAM's automatic mask generator needs a CUDA GPU to be practical — run this in
**Google Colab (T4)**. Pure-SAM is meaningless without SAM, so with no backend this notebook
**fails loudly** at the segmentation step (it never fabricates rooms).

## 1 — Get the project onto the Colab runtime (git clone)
The cell below clones the repo onto the Colab GPU machine, so `import scan2bim` and the
Stage-1 rasters (`scan2bim_out/stage1_occupancy.zip`, versioned in the repo — 247 KB) are
available with **no Drive copy and no manual upload**. Re-running is safe: it `git pull`s an
existing clone instead of re-cloning. Locally (not Colab) it just imports the editable
install. **Set `BRANCH`** to the branch holding the pure-SAM method.

> Why clone and not Drive? It is the most reproducible / easy-to-track path: the notebook
> always runs the exact code on the branch, nothing drifts. The big point clouds are **not**
> needed here — this stage only reads the Stage-1 rasters from the bundled ZIP.

In [ ]:
# ============== scan2bim setup (Colab via git clone) — the ONLY paths cell ==============
import sys, os
import numpy as np

REPO_URL = 'https://github.com/PrinceofJ/ONESTRUCTION-Point-Cloud-to-BIM.git'
BRANCH   = 'sam-auto-method'                 # the branch holding the pure-SAM method
CLONE    = '/content/onestruction'

try:
    import google.colab            # noqa: F401  (only importable on Colab)
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir(CLONE):
        get_ipython().system(f'git clone --depth 1 -b {BRANCH} {REPO_URL} {CLONE}')
    else:
        get_ipython().system(f'cd {CLONE} && git pull --ff-only')
    os.chdir(CLONE)
    if CLONE not in sys.path:
        sys.path.insert(0, CLONE)
    import scan2bim
    PROJECT_DIR = CLONE
else:                                        # local: the editable install resolves it
    import scan2bim
    PROJECT_DIR = scan2bim.project_root()

from scan2bim import artifacts as A
OUT_ROOT = os.path.join(PROJECT_DIR, 'scan2bim_out')
print('scan2bim', scan2bim.__version__, '| in_colab:', IN_COLAB)
print('PROJECT_DIR:', PROJECT_DIR)
print('OUT_ROOT   :', OUT_ROOT,
      '| stage1 zip present:', os.path.isfile(A.stage_zip(OUT_ROOT, A.STAGE1)))

## 2 — Check the GPU, then install SAM 2 + the SAM 2.1 checkpoint
SAM 2's automatic mask generator needs a CUDA GPU. In Colab: **Runtime > Change runtime
type > GPU**. Without a working SAM backend the segmentation step raises a clear error
(pure-SAM does not pass through like the refinement stage).

In [ ]:
# ------------------------------- GPU check -------------------------------
try:
    import torch
    if torch.cuda.is_available():
        print('CUDA OK ->', torch.cuda.get_device_name(0))
    else:
        print('WARNING: no CUDA GPU. In Colab: Runtime > Change runtime type > GPU. '
              'Without a SAM backend the segmentation step will raise.')
except Exception as e:
    print('torch not installed yet:', e, '- install it in the next cell (Colab).')

In [ ]:
# ===== Install SAM 2 + download the SAM 2.1 checkpoint (Colab; runs itself, idempotent) =====
# Verified against github.com/facebookresearch/sam2. Needs python>=3.10, torch>=2.5.1,
# torchvision>=0.20.1 (preinstalled on Colab GPU runtimes). Installs once if missing and
# downloads the checkpoint once if absent. SAME backend/checkpoint as the refinement stage.
SAM_CKPT = '/content/sam2.1_hiera_large.pt'
SAM_CFG  = 'configs/sam2.1/sam2.1_hiera_l.yaml'
CKPT_URL = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt'

if IN_COLAB:
    import importlib.util
    if importlib.util.find_spec('sam2') is None:
        get_ipython().system('pip install -q "git+https://github.com/facebookresearch/sam2.git"')
    if not os.path.exists(SAM_CKPT):
        get_ipython().system('wget -q -O {SAM_CKPT} {CKPT_URL}')
    print('SAM 2 ready | checkpoint:', SAM_CKPT, '| exists:', os.path.exists(SAM_CKPT))
else:
    print('Local run - skipping SAM install; the segmentation step will raise without a backend.')
print('SAM config name:', SAM_CFG)

## 3 — Load CFG (validated against Stage 1) + the Stage-1 rasters
`CFG` comes from the unified `load_config()` (params.yaml) and is **validated** against the
`config.json` saved inside the Stage-1 ZIP, so the SAM labels land on the exact grid the
other two methods use. Only the runtime / SAM fields (paths + backend) are overridden. The
paper's Table-1 parameters live in `params.yaml` / `Config` (`sam_points_per_side`, etc.).

In [ ]:
from PIL import Image
s1 = A.load_stage_dir(OUT_ROOT, A.STAGE1)               # shared Stage-1 occupancy rasters

CFG = scan2bim.load_config(start=PROJECT_DIR, out_root=OUT_ROOT)
scan2bim.assert_upstream_config(CFG, A.load_stage_config(s1))   # same cloud + grid as Stage 1

# runtime / SAM overrides only (do NOT touch geometry params)
CFG.sam_backend   = 'sam2'                              # 'sam2' | 'sam3' | 'sam1'
CFG.sam_ckpt      = SAM_CKPT
CFG.sam_model_cfg = SAM_CFG
SHOW_DEBUG = True

occ      = np.array(Image.open(os.path.join(s1, A.OCC_PNG)))
wallness = A.load_npy(os.path.join(s1, A.WALLNESS_NPY)).astype(bool)
coverage = A.load_npy(os.path.join(s1, A.COVERAGE_NPY)).astype(bool)
tf       = A.load_transform(os.path.join(s1, A.TRANSFORM_JSON))
print('raster (H x W):', wallness.shape, '| pixel_m =', CFG.pixel_m,
      '| points_per_side =', CFG.sam_points_per_side,
      '| A =', CFG.sam_auto_min_room_area_m2, 'm^2')

## 4 — Build the SAM image and run automatic "segment everything"
`build_sam_image` stacks occupancy + wallness + coverage as channels (the same image the
prompted refinement uses). `segment_rooms_sam_auto` runs SAM's automatic mask generator,
turns masks into rooms (room/not-room by area `A`, deterministic overlap resolution, walls
re-imposed as `-1`), and — if enabled in `params.yaml` — does the corridor reprocessing
(`sam_reprocess_residual`) and/or boundary buffer (`sam_auto_buffer_rooms`). **Pure-SAM
needs a real backend; if none built, we stop here rather than emit an empty map.**

In [ ]:
image = scan2bim.build_sam_image(occ, wallness, coverage, mode=CFG.sam_image_mode)

# walls scaffold = the wallness raster (consistent with the boundary-ring wall assignment's
# wall_source='wallness'); coverage drops the exterior background mask SAM always emits.
labels, dbg = scan2bim.segment_rooms_sam_auto(image, walls=wallness, coverage=coverage, cfg=CFG)
print(dbg)
if not dbg['ran']:
    raise RuntimeError(
        'Pure-SAM produced no segmentation: ' + dbg.get('reason', 'no SAM backend') +
        '. This method REQUIRES a SAM backend (run on a Colab GPU with the checkpoint '
        'installed above); it does not fall back like the refinement stage.')
print('backend', dbg['backend'], '| masks', dbg['n_masks'], '-> kept', dbg['n_kept'],
      '(void-dropped', dbg['n_void_dropped'], ') | rooms pass1', dbg['n_rooms_pass1'],
      '-> out', dbg['n_rooms_out'], '| reprocess +', dbg['reprocess']['n_added'])

## Optional — QA plot (SAM input image vs the resulting rooms)

In [ ]:
if SHOW_DEBUG:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    ax[0].imshow(image); ax[0].set_title('SAM input (occupancy / wallness / coverage)')
    rgb = np.ones(labels.shape + (3,))
    rgb[labels == -1] = (0, 0, 0); rgb[labels == 0] = (0.93, 0.93, 0.93)
    cmap = plt.get_cmap('tab20')
    rooms = [int(x) for x in np.unique(labels) if x >= 1]
    for k, r in enumerate(rooms):
        rgb[labels == r] = cmap(k % 20)[:3]
    ax[1].imshow(rgb); ax[1].set_title('SAM rooms: %d' % len(rooms))
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()

## 5 — Save room_labels on the Stage-1 grid + package the stage
Writes to `stage_sam_auto/` (distinct from the other methods, so nothing clobbers). Emitting
`room_labels.npy` on the Stage-1 grid lets `notebook_2_wall_assignment.ipynb` and
`evaluation/pq_eval.ipynb` consume it by name, exactly like the watershed output.

In [ ]:
out_dir = A.ensure_dir(A.stage_dir(OUT_ROOT, A.STAGE_SAM_AUTO))
A.save_npy(os.path.join(out_dir, A.ROOM_LABELS_NPY), labels.astype('int32'))
A.save_label_png(os.path.join(out_dir, A.ROOM_LABELS_PNG), labels)
A.save_transform(os.path.join(out_dir, A.TRANSFORM_JSON), tf)
A.save_config(os.path.join(out_dir, A.CONFIG_JSON), CFG)
zip_path = A.package_stage(OUT_ROOT, A.STAGE_SAM_AUTO)
print('packaged ->', zip_path)

## Optional — download stage_sam_auto.zip (Colab)
After this finishes, download `stage_sam_auto.zip`, place it in your local `scan2bim_out/`,
then run `notebook_2_wall_assignment.ipynb` to assign walls and back-project to 3-D.

In [ ]:
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not in Colab - find the ZIP at:', zip_path)